> ### ▶ Running this notebook in Google Colab (recommended)> 1. Go to [colab.research.google.com](https://colab.research.google.com)> 2. Click **File → Upload notebook**> 3. Select this `.ipynb` file from your downloaded course ZIP> 4. Click **Runtime → Run all**>> Alternatively: upload the whole ZIP to **Google Drive**, then double-click any notebook — it opens in Colab automatically.>> _No installation required. All you need is a free Google account._

## 4.1 — Setup

In [ ]:
import sysIN_COLAB = 'google.colab' in sys.modulesif IN_COLAB:    import subprocess    subprocess.run([sys.executable, '-m', 'pip', 'install', 'openpyxl', '--quiet'])import pandas as pdimport numpy as npimport matplotlib.pyplot as pltprint("✅ Ready")

---## 4.2 — Model Inputs

In [ ]:
# ── DCF ASSUMPTIONS ──────────────────────────────────────────────────────────company = "Fictional Co. Ltd"# Operating assumptionsbase_revenue     = 10_000   # £000srevenue_growth   = [0.12, 0.10, 0.09, 0.08, 0.07]  # 5-year forecastebitda_margin    = 0.22da_pct_revenue   = 0.04capex_pct_rev    = 0.05nwc_change_pct   = 0.01     # Working capital increase as % of revenue growthtax_rate         = 0.25# WACC componentscost_of_equity   = 0.10     # CAPM-derivedcost_of_debt     = 0.045debt_weight      = 0.30equity_weight    = 0.70# Terminal valueterminal_growth  = 0.025    # Long-run nominal GDP growth# Equity bridgenet_debt         = 1_500    # £000s (debt minus cash)shares_outstanding = 5_000  # thousandsyears = list(range(1, 6))print(f"DCF model for: {company}")print(f"Forecast period: {len(years)} years")

---## 4.3 — Free Cash Flow Projection

In [ ]:
# Build FCF projectionfcf_data = {}rev = base_revenuerevenues, ebitdas, ebits, taxes, das, capexs, nwcs, fcfs = [], [], [], [], [], [], [], []for i, g in enumerate(revenue_growth):    prev_rev = rev    rev = rev * (1 + g) if i > 0 else rev    ebitda  = rev * ebitda_margin    da      = rev * da_pct_revenue    ebit    = ebitda - da    tax     = ebit * tax_rate    nopat   = ebit - tax    capex   = rev * capex_pct_rev    delta_rev = rev - prev_rev if i > 0 else 0    nwc_chg = delta_rev * nwc_change_pct    fcf     = nopat + da - capex - nwc_chg    revenues.append(rev); ebitdas.append(ebitda); ebits.append(ebit)    taxes.append(tax); das.append(da); capexs.append(capex)    nwcs.append(nwc_chg); fcfs.append(fcf)proj = pd.DataFrame({    'Revenue':    revenues,    'EBITDA':     ebitdas,    'EBIT':       ebits,    'Tax':        taxes,    'D&A':        das,    'CapEx':      capexs,    'NWC Change': nwcs,    'Free Cash Flow': fcfs}, index=[f'Year {y}' for y in years]).round(1)proj

---## 4.4 — WACC and Discounting

In [ ]:
# WACCwacc = (equity_weight * cost_of_equity) + (debt_weight * cost_of_debt * (1 - tax_rate))print(f"WACC: {wacc*100:.2f}%")# Discount factorsdiscount_factors = [(1 / (1 + wacc) ** t) for t in years]# PV of forecast FCFspv_fcfs = [fcf * df for fcf, df in zip(fcfs, discount_factors)]pv_fcf_total = sum(pv_fcfs)print(f"PV of forecast FCFs: £{pv_fcf_total:,.0f}k")# Terminal value (Gordon Growth Model)terminal_fcf = fcfs[-1] * (1 + terminal_growth)terminal_value = terminal_fcf / (wacc - terminal_growth)pv_terminal = terminal_value * discount_factors[-1]print(f"Terminal Value:       £{terminal_value:,.0f}k")print(f"PV of Terminal Value: £{pv_terminal:,.0f}k")# Enterprise and Equity Valueenterprise_value = pv_fcf_total + pv_terminalequity_value     = enterprise_value - net_debtprice_per_share  = equity_value / shares_outstandingprint()print("=" * 40)print(f"Enterprise Value: £{enterprise_value:,.0f}k")print(f"Equity Value:     £{equity_value:,.0f}k")print(f"Implied Price:    £{price_per_share:.2f} per share")print("=" * 40)

---## 4.5 — Sensitivity: Implied Share Price vs WACC & Terminal Growth

In [ ]:
# Sensitivity table — implied share pricewaccs   = [0.08, 0.09, 0.10, 0.11, 0.12]tgrowths = [0.015, 0.020, 0.025, 0.030, 0.035]rows = {}for w in waccs:    row = {}    for tg in tgrowths:        dfs    = [(1/(1+w)**t) for t in years]        tv     = (fcfs[-1]*(1+tg)) / (w - tg)        pv_tv  = tv * dfs[-1]        ev     = sum(f*d for f,d in zip(fcfs,dfs)) + pv_tv        eq     = ev - net_debt        price  = eq / shares_outstanding        row[f"TG {tg*100:.1f}%"] = round(price, 2)    rows[f"WACC {w*100:.0f}%"] = rowsensitivity = pd.DataFrame(rows).Tprint("Implied Share Price (£) — WACC vs Terminal Growth Rate")print("Base case highlighted: WACC 10%, TG 2.5%")sensitivity

In [ ]:
# Visualise the sensitivity as a heatmapfig, ax = plt.subplots(figsize=(9, 5))data = sensitivity.values.astype(float)im = ax.imshow(data, cmap='RdYlGn', aspect='auto')ax.set_xticks(range(len(sensitivity.columns)))ax.set_yticks(range(len(sensitivity.index)))ax.set_xticklabels(sensitivity.columns, fontsize=9)ax.set_yticklabels(sensitivity.index, fontsize=9)ax.set_title(f"DCF Sensitivity — Implied Share Price (£)\n{company}", fontsize=11)for i in range(data.shape[0]):    for j in range(data.shape[1]):        ax.text(j, i, f"£{data[i,j]:.2f}", ha='center', va='center', fontsize=8,                color='black', fontweight='bold' if (i==2 and j==2) else 'normal')plt.colorbar(im, ax=ax, label='Implied Share Price (£)')plt.tight_layout()plt.show()

---## 4.6 — Key Takeaways- All assumptions in one block — change WACC or terminal growth and re-run in seconds- The FCF loop is more transparent than a row of Excel formulas that reference each other laterally- The sensitivity heatmap replaces a colour-coded Excel table — and it's generated automatically- The base case is trivially easy to identify: it's just a named cell in your assumptions dict- This model is now a reusable function: pass in different company data and get a new DCF instantly---## Up Next: Module 5 — Financial Modelling Best PracticesVersion control with Git, structuring reusable functions, and building model templates you can use across projects.